In [1]:
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)   



c:\Users\Labdhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4149.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
from langchain_chroma import Chroma

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="db/chroma_db",  
)

In [3]:
query="where is hilti based on"

In [4]:
results=vector_store.similarity_search(query,k=10)

In [5]:
for res in results:
    print(res.page_content)

Hilti, a registered trademark of the various Hilti corporate entities, is the family name of the company's founders.
Hilti Corporation (Hilti Aktiengesellschaft or Hilti AG; also known as Hilti Group) is a Liechtenstein multinational company that develops, manufactures, and markets products for the construction, building maintenance, energy and manufacturing industries, mainly to the professional end-user. It concentrates mainly on anchoring systems, fire protection systems, installation systems, measuring and detection tools (such as laser levels, range meters and line lasers), power tools (such as hammer drills, demolition hammers, diamond drills, cordless electric drills, heavy angle drills, power saws) and related software and services.[3]
A Hilti store in Hong Kong Hilti is based in Schaan, Liechtenstein, and is the principality's largest employer. The company employs around 30,000 people worldwide.[2]
Hilti North America (HNA)
At this time Hilti opened offices in Italy, Belgium, 

In [6]:
context_for_llm=context = "\n\n".join([doc.page_content for doc in results])


In [7]:
prompt=f"""
Answer the question based on the given context.
context: {context_for_llm}
question: {query}
"""

In [8]:
import sys
!{sys.executable} -m pip install groq



[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
from dotenv import load_dotenv
load_dotenv() 
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt}
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Hilti is based in Schaan, Liechtenstein.


Using local llm:

In [10]:
import sys
!{sys.executable} -m pip install ollama


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from ollama import chat

messages = [
     {
         "role": "user",
        "content": prompt,
    },
 ]

response = chat(model="llama3.1:8b", messages=messages)
print(response.message.content)


Hilti is based in Schaan, Liechtenstein.


In [12]:
def chatbot():
    while True:
        user_query=input("Enter the query to this chatbot:")
        if(user_query=="quit"):
            break
        results=vector_store.similarity_search(user_query,k=10)
        context_for_llm = "\n\n".join([doc.page_content for doc in results])
        prompt=f"""
        Answer the question based on the given context.
        context: {context_for_llm}
        question: {user_query}
        """
        chat_completion = client.chat.completions.create(
        messages=[
            {"role": "user", "content": prompt}
        ],
        model="llama-3.3-70b-versatile",
    )
        print(chat_completion.choices[0].message.content)
        print("This message was generated")

In [ ]:
chatbot()